In [1]:
import numpy as np
import pandas as pd

#### Read and explore the given dataset.  ( Rename column/add headers, plot histograms, find data characteristics)

In [2]:
data = pd.read_csv('ratings_Electronics.csv',names=['userId', 'productId', 'ratings','timestamp'])

In [3]:
data.head()

,userId,productId,ratings,timestamp
0,AKM1MP6P0OYPR,0132793040,5.0,1365811200
1,A2CX7LUOHB2NDG,0321732944,5.0,1341100800
2,A2NWSAGRHCP8N5,0439886341,1.0,1367193600
3,A2WNBOD3WNDNKT,0439886341,3.0,1374451200
4,A1GI0U4ZRJA8WN,0439886341,1.0,1334707200


In [4]:
data.shape

(7824482, 4)

In [5]:
data.hist()

array([[<matplotlib.axes._subplots.AxesSubplot object at 0x11915f898>,
      dtype=object)

#### Take a subset of the dataset to make it less sparse/ denser. ( For example, keep the users only who has given 50 or more number of ratings )


In [11]:
# timestamp is not important so discarding it
data = data.drop('timestamp', axis=1)

In [15]:
counts = data['userId'].value_counts()
data_final = data[data['userId'].isin(counts[counts >= 50].index)]

In [16]:
print('Number of users who has given 50 or more number of ratings:', len(data_final))


Number of users who has given 50 or more number of ratings: 125871


In [18]:
print('Number of unique users:', data_final['userId'].nunique())


Number of unique users: 1540


In [20]:
print('Number of unique products: ', data_final['productId'].nunique())

Number of unique products:  48190


#### Split the data randomly into train and test dataset. ( For example, split it in 70/30 ratio)

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.externals import joblib
import Recommenders as Recommenders
import Evaluation as Evaluation

In [13]:
train_data, test_data = train_test_split(data, test_size = 0.30, random_state=0)
print(train_data.head(5))

                 userId   productId  ratings
5258360  A1H898ODS23YBE  B0060I17XA      5.0
4191577  A3OATVQ0ZPA0O9  B004J3Y9U6      3.0
5574835  A10F3XNIDFZK08  B0073HSHVM      3.0
1619920   AOEAD7DPLZE53  B0012W7HQK      5.0
424298   A2T5K3LO6TQMQW  B00022OBOM      4.0


In [21]:
#Count of user_id for each unique product as recommendation score 
train_data_grouped = train_data.groupby('productId').agg({'userId': 'count'}).reset_index()
train_data_grouped.rename(columns = {'userId': 'score'},inplace=True)
train_data_grouped.head()

,productId,score
0,0439886341,3
1,0511189877,4
2,0528881469,20
3,0558835155,1
4,059400232X,3


#### Build Popularity Recommender model.

In [24]:
#Sort the products on recommendation score 
train_data_sort = train_data_grouped.sort_values(['score', 'productId'], ascending = [0,1]) 
      
#Generate a recommendation rank based upon score 
train_data_sort['Rank'] = train_data_sort['score'].rank(ascending=0, method='first') 
          
#Get the top 5 recommendations 
popularity_recommendations = train_data_sort.head(5) 
popularity_recommendations 

,productId,score,Rank
271590,B0074BW614,12730,1.0
375363,B00DR0PDNE,11499,2.0
287933,B007WTAJTO,9994,3.0
261254,B006GWO5WK,8567,4.0
91788,B0019EHU8G,8547,5.0


In [25]:
users = data['userId'].unique()

In [29]:
# Use popularity based recommender model to make predictions
def recommend(user_id):     
    user_recommendations = popularity_recommendations 
          
    #Add user_id column for which the recommendations are being generated 
    user_recommendations['userId'] = user_id 
      
    #Bring user_id column to the front 
    cols = user_recommendations.columns.tolist() 
    cols = cols[-1:] + cols[:-1] 
    user_recommendations = user_recommendations[cols] 
          
    return user_recommendations 

In [30]:
find_recom = [10,125,200]   # This list is user choice.
for i in find_recom:
    print("Here is the recommendation for the userId: %d\n" %(i))
    print(recommend(i))    
    print("\n") 

Here is the recommendation for the userId: 10

        userId   productId  score  Rank
271590      10  B0074BW614  12730   1.0
375363      10  B00DR0PDNE  11499   2.0
287933      10  B007WTAJTO   9994   3.0
261254      10  B006GWO5WK   8567   4.0
91788       10  B0019EHU8G   8547   5.0


Here is the recommendation for the userId: 125

        userId   productId  score  Rank
271590     125  B0074BW614  12730   1.0
375363     125  B00DR0PDNE  11499   2.0
287933     125  B007WTAJTO   9994   3.0
261254     125  B006GWO5WK   8567   4.0
91788      125  B0019EHU8G   8547   5.0


Here is the recommendation for the userId: 200

        userId   productId  score  Rank
271590     200  B0074BW614  12730   1.0
375363     200  B00DR0PDNE  11499   2.0
287933     200  B007WTAJTO   9994   3.0
261254     200  B006GWO5WK   8567   4.0
91788      200  B0019EHU8G   8547   5.0




/Users/kumanish/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  


#### Build Collaborative Filtering model.

In [31]:
df_collaborative_filter = pd.concat([train_data, test_data]).reset_index()
c.tail()

,index,userId,productId,ratings
7824477,6241487,AW2BYVGYR57O,B008HY8XTG,5.0
7824478,4356226,A35456I12N9KHW,B004Q81CKY,5.0
7824479,7333376,A2PJZ8FGZC48YG,B00CYZYKXW,4.0
7824480,2613787,A39CX70TRCL3KM,B002HWRJBM,5.0
7824481,808854,A1MT815819FZC5,B000CP4K8Q,2.0


In [ ]:
#User-based Collaborative Filtering
pivot_df = df_collaborative_filter.pivot(index = 'userId', columns ='productId', values = 'ratings').fillna(0)
print(pivot_df.shape)
pivot_df.head()

In [ ]:
pivot_df['user_index'] = np.arange(0, pivot_df.shape[0], 1)
pivot_df.head()

In [ ]:
pivot_df.set_index(['user_index'], inplace=True)

# Actual ratings given by users
pivot_df.head()

In [ ]:
from scipy.sparse.linalg import svds
# Singular Value Decomposition
U, sigma, Vt = svds(pivot_df, k = 50)
# Construct diagonal array in SVD
sigma = np.diag(sigma)